# İSTANBUL TOPKAPI ÜNİVERSİTESİ
## DERİN ÖĞRENME ARASINAV ÖDEVİ
## Hüseyin Yunus Demir 25221001017

**Konu:** Yüz Maskesi Tespiti (Face Mask Detection)
**Model:** DenseNet121 (Pre-trained)

### 1. Gerekli Kütüphaneler

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import tensorflow as tf
from google.colab import drive
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback

# TensorFlow sürümünü
print(f"TensorFlow Version: {tf.__version__}")


### 2. Kaggle Veri Setini İndirme ve Hazırlama
Bu adımda, Kaggle API kullanarak `Face Mask Detection Dataset`.

In [ ]:
import os

# Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Veri setini indirme
!kaggle datasets download -d omkargurav/face-mask-dataset

# İndirilen zip dosyasını açma (-o to overwrite without prompting)
!unzip -o -q face-mask-dataset.zip -d face_mask_dataset

# Veri setinin yapısını kontrol etmek için
print("Contents of face_mask_dataset after unzipping:")
!ls -R face_mask_dataset/

# Veri setinin yapısını kontrol etme
base_dir = "face_mask_dataset/data/" # Corrected path

# Verify if the directory exists before proceeding
if not os.path.exists(base_dir):
    print(f"Error: The directory '{base_dir}' was not found by Python. Please check the `!ls -R face_mask_dataset/` output for the exact directory structure after unzipping.")
    raise FileNotFoundError(f"Base directory not found: '{base_dir}'")
else:
    print(f"Success: The directory '{base_dir}' found by Python.")

for dir_name in os.listdir(base_dir):
    print(f"Klasör: {dir_name}, İçerik: {os.listdir(os.path.join(base_dir, dir_name))}")

### 3. Veri Setinin Eğitim/Validasyon ve Test Olarak Ayrılması ve Ön İşleme

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 100

# Veri artırımı ve ön işleme
datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

# Veri setini yükleme ve ayırma
train_directory_for_generators = base_dir
test_directory_for_generators = base_dir

train_generator = datagen.flow_from_directory(
    train_directory_for_generators,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42
)

validation_generator = datagen.flow_from_directory(
    train_directory_for_generators,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42
)

test_datagen = ImageDataGenerator(rescale=1.0/255)
test_generator = test_datagen.flow_from_directory(
    test_directory_for_generators,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=42
)

### 4. Model Mimarisi (DenseNet121)

In [ ]:
def build_model():
    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dense(256, activation='relu')(x)
    x = Dense(128, activation='relu')(x)
    x = Dense(64, activation='relu')(x)
    predictions = Dense(2, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model

# Kayıtlı model varsa onu yükle, yoksa yeni model oluştur
if os.path.exists(LAST_MODEL_PATH):
    print('Kayıtlı son model bulundu, yükleniyor...')
    model = load_model(LAST_MODEL_PATH)
else:
    print('Yeni model oluşturuluyor...')
    model = build_model()
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()


### 5. Model Eğitimi ve Callbacks

In [ ]:
class SaveTrainingState(Callback):
    def __init__(self, state_path, history_path, last_model_path):
        super().__init__()
        self.state_path = state_path
        self.history_path = history_path
        self.last_model_path = last_model_path
        self.history_log = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

        if os.path.exists(self.history_path):
            try:
                with open(self.history_path, 'r', encoding='utf-8') as f:
                    loaded = json.load(f)
                    if isinstance(loaded, dict):
                        self.history_log.update(loaded)
            except Exception as e:
                print('Geçmiş geçmişi okunamadı:', e)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for key in self.history_log.keys():
            value = logs.get(key)
            if value is not None:
                self.history_log[key].append(float(value))

        with open(self.state_path, 'w', encoding='utf-8') as f:
            json.dump({'last_finished_epoch': epoch + 1}, f, ensure_ascii=False, indent=2)

        with open(self.history_path, 'w', encoding='utf-8') as f:
            json.dump(self.history_log, f, ensure_ascii=False, indent=2)

        self.model.save(self.last_model_path)
        print(f"Epoch {epoch + 1} tamamlandı, durum Drive'a kaydedildi.")

initial_epoch = 0
if os.path.exists(STATE_PATH):
    try:
        with open(STATE_PATH, 'r', encoding='utf-8') as f:
            initial_epoch = json.load(f).get('last_finished_epoch', 0)
        print('Devam edilecek epoch:', initial_epoch)
    except Exception as e:
        print('State dosyası okunamadı, 0. epochtan başlanacak:', e)
        initial_epoch = 0

early_stopping = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, min_lr=1e-6)
model_checkpoint = ModelCheckpoint(BEST_MODEL_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
save_state_callback = SaveTrainingState(STATE_PATH, HISTORY_PATH, LAST_MODEL_PATH)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    validation_data=validation_generator,
    callbacks=[early_stopping, reduce_lr, model_checkpoint, save_state_callback]
)


### 6. Performans Analizi ve Grafikler

In [ ]:
# History verisini birleştirerek kullan
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH, 'r', encoding='utf-8') as f:
        saved_history = json.load(f)
else:
    saved_history = history.history

# Eğitim ve Doğrulama Grafikleri
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(saved_history.get('accuracy', []), label='Eğitim Doğruluğu')
plt.plot(saved_history.get('val_accuracy', []), label='Doğrulama Doğruluğu')
plt.title('Model Doğruluğu')
plt.xlabel('Epok')
plt.ylabel('Doğruluk')
plt.legend()
plt.show()

plt.subplot(1, 2, 2)
plt.plot(saved_history.get('loss', []), label='Eğitim Kaybı')
plt.plot(saved_history.get('val_loss', []), label='Doğrulama Kaybı')
plt.title('Model Kaybı')
plt.xlabel('Epok')
plt.ylabel('Kayıp')
plt.legend()
plt.show()

# Test seti üzerinde tahmin yapma
if os.path.exists(BEST_MODEL_PATH):
    model = load_model(BEST_MODEL_PATH)
    print('En iyi model Drive üzerinden yüklendi.')
else:
    print('En iyi model bulunamadı, mevcut model kullanılacak.')

y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

# Karmaşıklık Matrisi (Confusion Matrix)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=test_generator.class_indices.keys(), yticklabels=test_generator.class_indices.keys())
plt.title('Karmaşıklık Matrisi')
plt.ylabel('Gerçek Etiket')
plt.xlabel('Tahmin Edilen Etiket')
plt.show()

# ROC Eğrisi ve AUC
fpr, tpr, _ = roc_curve(y_true, y_pred_probs[:, 1])
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC eğrisi (alan = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('ROC Eğrisi')
plt.legend(loc="lower right")
plt.show()

# Diğer Metrikler
print("
Sınıflandırma Raporu:")
print(classification_report(y_true, y_pred, target_names=list(test_generator.class_indices.keys())))
